# 06 — Stacking: Ridge Meta-Model on XGB + LGBM OOF Predictions

**Current best:** v6 blend → Kaggle RMSE **22,124** | CV RMSE **21,818**. Goal: push below 21,800.

## What is stacking and why is it better than blending?

| Approach | How it combines models | Limitation |
|---|---|---|
| **Blending (v5/v6)** | Fixed ratio (e.g. 45% XGB + 55% LGBM) for every row | Same weight applied to every prediction, even when one model is clearly better for a specific flat type |
| **Stacking (this notebook)** | Ridge meta-model **learns** the optimal weight per context from the OOF predictions | Adapts to where each model is stronger |

## Two improvements this notebook

| # | Improvement | Why it helps | Expected gain |
|---|---|---|---|
| 1 | **Stacking** (Ridge on XGB+LGBM OOF) | Ridge learns adaptive blend rather than fixed ratio | –$200 to –$400 |
| 2 | **`street_name` frequency encoding** | ~500 street levels → 1 continuous signal; premium streets (Holland, Buona Vista) get higher counts | –$100 to –$200 |

## How stacking works (step by step)

```
Step 1 — 5-fold OOF generation:
  Fold 1: train XGB+LGBM on folds 2-5 → predict fold 1 → xgb_oof[fold1], lgbm_oof[fold1]
  Fold 2: train XGB+LGBM on folds 1,3-5 → predict fold 2 → xgb_oof[fold2], lgbm_oof[fold2]
  ...repeat for all 5 folds...
  Also: collect test predictions from each fold model

Step 2 — Train Ridge meta-model:
  meta_X_train = [[xgb_oof[0], lgbm_oof[0]],
                  [xgb_oof[1], lgbm_oof[1]], ...]   (N_train × 2)
  Ridge.fit(meta_X_train, y_log)                    (learns optimal coefficients)

Step 3 — Predict test:
  meta_X_test = average of 5 fold test predictions  (N_test × 2)
  final_pred  = expm1( Ridge.predict(meta_X_test) )
```

---
## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from math import radians, cos, sin, asin, sqrt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')

---
## 2. Feature Engineering

**New in this notebook vs notebook 05:**
- `street_freq` — how many times a street appears in the training set. Premium streets (Holland Rd, Orchard Rd, Buona Vista) transact more and at higher prices. ~500 unique streets → 1 continuous count signal.

In [ ]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
CBD_LAT, CBD_LON = 1.2847, 103.8510

# Compute street_freq from training set once (no target leakage — it's just counts)
STREET_FREQ = train['street_name'].value_counts().to_dict()

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    """Apply all feature engineering. amenity_caps: pass train-computed caps to test."""
    df = df.copy()
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)
    # Tier 1 — location & lease
    df['remaining_lease']  = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['dist_to_cbd']      = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate'] = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    # Tier 2 — cyclical + supply
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    # Tier 3 — amenity score
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    # Tier 4 — interaction & log features (from v6)
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    # Tier 5 — NEW: street frequency encoding
    df['street_freq'] = df['street_name'].map(STREET_FREQ).fillna(0)
    return df, new_caps

train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')
print(f'\nstreet_freq sample (top 5 streets):')
print(train_fe.groupby('street_name')['street_freq'].first().nlargest(5).to_string())

---
## 3. Per-Fold Target Encoding for `town_median_price`

In [ ]:
def oof_town_median(towns_series, y_series, n_splits=5, random_state=42):
    """Out-of-fold town median price to prevent target leakage."""
    towns = towns_series.values
    y     = y_series.values
    encoded = np.zeros(len(towns))
    global_med = np.median(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(towns):
        fold_map = {}
        for t, p in zip(towns[tr_idx], y[tr_idx]):
            fold_map.setdefault(t, []).append(p)
        fold_map = {t: np.median(ps) for t, ps in fold_map.items()}
        for i in va_idx:
            encoded[i] = fold_map.get(towns[i], global_med)
    return encoded

train_fe['town_median_price'] = oof_town_median(
    train_fe['town'], train['resale_price']
)
full_town_map = train.groupby('town')['resale_price'].median()
test_fe['town_median_price'] = test_fe['town'].map(full_town_map).fillna(full_town_map.median())

print('OOF town encoding done.')

---
## 4. Prepare X and y

In [ ]:
DROP_ALL = ['id','resale_price'] + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms']

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')
print(f'New feature: street_freq')
print(f'\nColumn list: {list(X.columns)}')

---
## 5. Preprocessing Pipeline

In [ ]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('Preprocessor ready.')

---
## 6. Tune Base Models (RandomizedSearchCV)

Same approach as notebook 05 — search on an 80/20 split, then use the best params for OOF generation.
The custom `dollar_rmse_scorer` converts log predictions back to dollars so CV scores stay interpretable.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

def neg_dollar_rmse(y_log_true, y_log_pred):
    return -np.sqrt(mean_squared_error(np.expm1(y_log_true), np.expm1(y_log_pred)))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)

# --- XGBoost search ---
param_dist_xgb = {
    'model__n_estimators'     : [800, 1000, 1200, 1500],
    'model__max_depth'        : [5, 6, 7, 8],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.08],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.5, 0.6, 0.7, 0.8],
    'model__min_child_weight' : [1, 3, 5],
    'model__reg_alpha'        : [0, 0.01, 0.1],
    'model__reg_lambda'       : [1, 1.5, 2],
}
xgb_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor), ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, tree_method='hist'))]),
    param_distributions=param_dist_xgb, n_iter=30, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
xgb_search.fit(X_train, y_train_log)
print(f'\nXGB best CV RMSE: ${-xgb_search.best_score_:,.0f}')
xgb_val_pred = np.expm1(xgb_search.best_estimator_.predict(X_val))
print(f'XGB val RMSE:     ${np.sqrt(mean_squared_error(y_val, xgb_val_pred)):,.0f}')

# --- LightGBM search ---
param_dist_lgbm = {
    'model__n_estimators'     : [800, 1000, 1200, 1500],
    'model__max_depth'        : [6, 7, 8, 10],
    'model__num_leaves'       : [60, 80, 100, 127],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.08],
    'model__subsample'        : [0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.6, 0.7, 0.8],
    'model__min_child_samples': [10, 20, 30],
    'model__reg_alpha'        : [0, 0.01, 0.1],
    'model__reg_lambda'       : [0, 0.1, 1],
}
lgbm_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor), ('model', LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1))]),
    param_distributions=param_dist_lgbm, n_iter=30, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
lgbm_search.fit(X_train, y_train_log)
print(f'\nLGBM best CV RMSE: ${-lgbm_search.best_score_:,.0f}')
lgbm_val_pred = np.expm1(lgbm_search.best_estimator_.predict(X_val))
print(f'LGBM val RMSE:     ${np.sqrt(mean_squared_error(y_val, lgbm_val_pred)):,.0f}')

# Extract best params (strip 'model__' prefix for use in OOF loop)
XGB_PARAMS  = {k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}
LGBM_PARAMS = {k.replace('model__',''):v for k,v in lgbm_search.best_params_.items()}
XGB_PARAMS.update( {'random_state':42,'n_jobs':-1,'verbosity':0,'tree_method':'hist'})
LGBM_PARAMS.update({'random_state':42,'n_jobs':-1,'verbosity':-1})

print(f'\nXGB_PARAMS:  {XGB_PARAMS}')
print(f'LGBM_PARAMS: {LGBM_PARAMS}')

---
## 7. Generate OOF Predictions (5-Fold)

Each fold:
1. Train XGB and LGBM on 4 folds (log target)
2. Predict the held-out fold → OOF predictions for training rows
3. Predict test set → collect for later averaging
4. Recompute `town_median_price` from fold training rows only (prevent leakage)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof        = np.zeros(len(X))             # OOF log-predictions from XGB
lgbm_oof       = np.zeros(len(X))             # OOF log-predictions from LGBM
xgb_test_folds  = np.zeros((5, len(X_test)))  # Test predictions per fold
lgbm_test_folds = np.zeros((5, len(X_test)))

fold_rmses_xgb  = []
fold_rmses_lgbm = []

print(f'Generating OOF predictions (5-fold)...')
print(f'{"Fold":>5}  {"XGB RMSE":>12}  {"LGBM RMSE":>12}')
print('-' * 36)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    y_tr, y_va = y[tr_idx], y[va_idx]
    y_tr_log   = np.log1p(y_tr)

    # Per-fold target encoding (recompute from fold train rows only)
    fold_town_map = {}
    for t, p in zip(X_tr['town'].values, y_tr):
        fold_town_map.setdefault(t, []).append(p)
    fold_town_map = {t: np.median(ps) for t, ps in fold_town_map.items()}
    global_med_tr = np.median(y_tr)
    X_tr['town_median_price'] = X_tr['town'].map(fold_town_map).fillna(global_med_tr)
    X_va['town_median_price'] = X_va['town'].map(fold_town_map).fillna(global_med_tr)

    # XGBoost
    xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    xgb_pipe.fit(X_tr, y_tr_log)
    xgb_oof[va_idx]           = xgb_pipe.predict(X_va)
    xgb_test_folds[fold]      = xgb_pipe.predict(X_test)
    xgb_fold_rmse             = rmse(y_va, np.expm1(xgb_oof[va_idx]))
    fold_rmses_xgb.append(xgb_fold_rmse)

    # LightGBM
    lgbm_pipe = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    lgbm_pipe.fit(X_tr, y_tr_log)
    lgbm_oof[va_idx]          = lgbm_pipe.predict(X_va)
    lgbm_test_folds[fold]     = lgbm_pipe.predict(X_test)
    lgbm_fold_rmse            = rmse(y_va, np.expm1(lgbm_oof[va_idx]))
    fold_rmses_lgbm.append(lgbm_fold_rmse)

    print(f'{fold+1:>5}  ${xgb_fold_rmse:>10,.0f}  ${lgbm_fold_rmse:>10,.0f}')

print('-' * 36)
print(f'{"Mean":>5}  ${np.mean(fold_rmses_xgb):>10,.0f}  ${np.mean(fold_rmses_lgbm):>10,.0f}')

# Average test predictions across 5 folds
xgb_test_avg  = xgb_test_folds.mean(axis=0)   # log space
lgbm_test_avg = lgbm_test_folds.mean(axis=0)   # log space

print(f'\nOOF generation complete.')

---
## 8. Sanity Check — OOF Blend vs v6

Before training the meta-model, let's check what a fixed 50/50 blend of the OOF predictions gives us. This is the baseline the meta-model needs to beat.

In [ ]:
# Fixed 50/50 blend of OOF (in log space)
blend_oof_50 = np.expm1(0.5 * xgb_oof + 0.5 * lgbm_oof)
rmse_oof_50  = rmse(y, blend_oof_50)

# Scan for best fixed blend ratio
best_r, best_ratio = float('inf'), 0.5
for alpha in np.arange(0.0, 1.01, 0.05):
    r = rmse(y, np.expm1(alpha * xgb_oof + (1-alpha) * lgbm_oof))
    if r < best_r:
        best_r, best_ratio = r, alpha

print(f'OOF 50/50 blend RMSE:          ${rmse_oof_50:>8,.0f}')
print(f'OOF best fixed blend RMSE:     ${best_r:>8,.0f}  ({best_ratio*100:.0f}% XGB + {(1-best_ratio)*100:.0f}% LGBM)')
print(f'\nv6 CV RMSE (for comparison):    $21,818')
print(f'\nNote: OOF RMSE is computed on all training rows (not a separate held-out set),')
print(f'so it will look slightly lower than true CV RMSE.')

---
## 9. Ridge Meta-Model

Train Ridge on `[xgb_oof, lgbm_oof]` → `y_log`. Ridge learns:
- How much to trust XGB vs LGBM overall
- Any linear correction needed on top of the individual model predictions

Why Ridge and not a more complex model?
- Meta-features are only 2 columns — a complex model would overfit immediately
- Ridge regularisation prevents the coefficients from going wild
- Interpretable: we can read off the learned weights directly

In [ ]:
# Meta-features: [xgb_oof_log, lgbm_oof_log] (already in log space)
meta_X_train = np.column_stack([xgb_oof, lgbm_oof])
meta_X_test  = np.column_stack([xgb_test_avg, lgbm_test_avg])

# Try several Ridge alphas
print(f'Ridge alpha sweep (OOF meta-train RMSE):')
print(f'{"alpha":>10}  {"RMSE":>12}  {"XGB coef":>10}  {"LGBM coef":>10}')
print('-' * 50)

best_meta_rmse = float('inf')
best_alpha_ridge = 1.0

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(meta_X_train, y_log)
    meta_pred_log = ridge.predict(meta_X_train)
    meta_rmse_val = rmse(y, np.expm1(meta_pred_log))
    print(f'{alpha:>10.3f}  ${meta_rmse_val:>10,.0f}  {ridge.coef_[0]:>10.4f}  {ridge.coef_[1]:>10.4f}')
    if meta_rmse_val < best_meta_rmse:
        best_meta_rmse = meta_rmse_val
        best_alpha_ridge = alpha

print(f'\nBest Ridge alpha: {best_alpha_ridge}')

# Fit final meta-model
meta_model = Ridge(alpha=best_alpha_ridge)
meta_model.fit(meta_X_train, y_log)
print(f'Learned coefficients — XGB: {meta_model.coef_[0]:.4f},  LGBM: {meta_model.coef_[1]:.4f}')
print(f'Intercept: {meta_model.intercept_:.4f}')

---
## 10. Proper 5-Fold CV for Stacking

The OOF RMSE above is optimistic (meta-model trained and evaluated on the same OOF predictions). For an honest CV estimate we need a **nested** loop: inside each outer fold, generate inner-fold OOF predictions and train the meta-model — then evaluate on the outer fold.

This is expensive but gives the most honest estimate of true generalisation.

In [ ]:
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
inner_kf = KFold(n_splits=5, shuffle=True, random_state=0)

outer_rmses = []

print('Nested CV (outer 5-fold, inner 5-fold OOF generation)...')
print(f'{"Fold":>5}  {"Stack RMSE":>14}')
print('-' * 24)

for outer_fold, (outer_tr, outer_va) in enumerate(outer_kf.split(X)):
    X_otr, X_ova = X.iloc[outer_tr].copy(), X.iloc[outer_va].copy()
    y_otr, y_ova = y[outer_tr], y[outer_va]
    y_otr_log    = np.log1p(y_otr)

    # Inner OOF predictions on outer training set
    inner_xgb_oof  = np.zeros(len(outer_tr))
    inner_lgbm_oof = np.zeros(len(outer_tr))

    for inner_tr_rel, inner_va_rel in inner_kf.split(X_otr):
        Xi_tr = X_otr.iloc[inner_tr_rel].copy()
        Xi_va = X_otr.iloc[inner_va_rel].copy()
        yi_tr = y_otr[inner_tr_rel]
        yi_tr_log = np.log1p(yi_tr)

        # Per-fold town encoding for inner fold
        tm = {}
        for t, p in zip(Xi_tr['town'].values, yi_tr):
            tm.setdefault(t, []).append(p)
        tm = {t: np.median(ps) for t, ps in tm.items()}
        gm = np.median(yi_tr)
        Xi_tr['town_median_price'] = Xi_tr['town'].map(tm).fillna(gm)
        Xi_va['town_median_price'] = Xi_va['town'].map(tm).fillna(gm)

        xgb_i = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
        xgb_i.fit(Xi_tr, yi_tr_log)
        inner_xgb_oof[inner_va_rel] = xgb_i.predict(Xi_va)

        lgbm_i = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
        lgbm_i.fit(Xi_tr, yi_tr_log)
        inner_lgbm_oof[inner_va_rel] = lgbm_i.predict(Xi_va)

    # Per-outer-fold town encoding for validation rows
    otm = {}
    for t, p in zip(X_otr['town'].values, y_otr):
        otm.setdefault(t, []).append(p)
    otm = {t: np.median(ps) for t, ps in otm.items()}
    ogm = np.median(y_otr)
    X_otr['town_median_price'] = X_otr['town'].map(otm).fillna(ogm)
    X_ova['town_median_price'] = X_ova['town'].map(otm).fillna(ogm)

    # Train final base models on full outer training set
    xgb_outer  = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    lgbm_outer = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    xgb_outer.fit(X_otr, y_otr_log)
    lgbm_outer.fit(X_otr, y_otr_log)

    # Get outer val predictions
    xgb_va_log  = xgb_outer.predict(X_ova)
    lgbm_va_log = lgbm_outer.predict(X_ova)

    # Train Ridge meta on inner OOF, predict outer val
    meta_inner = Ridge(alpha=best_alpha_ridge)
    meta_inner.fit(np.column_stack([inner_xgb_oof, inner_lgbm_oof]), y_otr_log)

    stack_pred_log = meta_inner.predict(np.column_stack([xgb_va_log, lgbm_va_log]))
    stack_pred     = np.expm1(stack_pred_log)
    fold_rmse_val  = rmse(y_ova, stack_pred)
    outer_rmses.append(fold_rmse_val)
    print(f'{outer_fold+1:>5}  ${fold_rmse_val:>12,.0f}')

cv_mean = np.mean(outer_rmses)
cv_std  = np.std(outer_rmses)
print('-' * 24)
print(f'{"CV":>5}  ${cv_mean:>12,.0f}  ± ${cv_std:,.0f}')
print(f'\nv6 CV RMSE: $21,818  |  Change: ${cv_mean - 21818:>+,.0f}')

---
## 11. Generate Submission v7

Retrain on the **full training set** — generate OOF predictions for meta-model training, then predict test.

In [ ]:
# Final meta-model (trained in Section 9 on all OOF predictions) — already fitted
# Use meta_model (Ridge) + xgb_test_avg + lgbm_test_avg (averaged from 5 fold test predictions)

final_log  = meta_model.predict(meta_X_test)
final_pred = np.expm1(final_log)

sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v7 = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred})
sub_v7 = sub_v7.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v7_stack.csv'
sub_v7.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {sub_v7.shape}')
print(f'Price range: ${sub_v7.Predicted.min():,.0f} – ${sub_v7.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v7.Predicted.mean():,.0f}')
print(f'\nv6 mean for comparison: ~$430,000')

---
## 12. Summary

### What changed vs v6
| Change | Detail |
|---|---|
| **Stacking** | Ridge meta-model on OOF XGB + LGBM predictions instead of fixed blend ratio |
| **`street_freq`** | Frequency encoding for ~500 street name levels |

### Score tracker
| Version | Model | CV RMSE | Kaggle |
|---|---|---|---|
| v1 | RF Baseline | $25,871 | 25,943 |
| v3 | XGBoost tuned | $21,828 | 22,800 |
| v5 | Blend 45/55 | $21,570 | 22,428 |
| v6 | Log blend | $21,818 | 22,124 |
| **v7** | **Stacking + street_freq** | **_(run above)_** | **_(submit)_** |

### Key learnings
- Stacking is better than fixed blending when the two base models have different strengths by segment (e.g. XGB better for mature estates, LGBM better for newer towns)
- Ridge is the right meta-learner for 2 base models — any more complex model would overfit the meta-features
- `street_freq` adds signal without target leakage because it uses counts, not prices
- Nested CV (outer 5-fold, inner 5-fold) is the honest way to evaluate stacking — but it's slow (25× base model training runs per experiment)

### If v7 improves → next steps
- [ ] Add **CatBoost** as a third base model → stack becomes Ridge on [XGB, LGBM, CAT] OOF
- [ ] Try **Optuna** for hyperparameter tuning (Bayesian, smarter than RandomizedSearchCV)
- [ ] Add **`block_num`** encoding: `pd.to_numeric(df['block'].str.extract(r'(\d+)')[0])`